In [1]:
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import scipy.stats as stats
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

Dir_github = '/work/desai-lab/xuanyang/Project/Semantic/dissemination/github/DiscoFMRI'
# Dir_github = r'Z:\study\phd\project\Semantics\FactorAnalysis\github'

Dir_output = os.path.join(Dir_github,'data','FactorAnalysis')

/home/xy6/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:

import numpy as np
from scipy.stats import t, norm
from math import atanh, pow
from numpy import tanh

def rz_ci(r, n, conf_level = 0.95):
    zr_se = pow(1/(n - 3), .5)
    moe = norm.ppf(1 - (1 - conf_level)/float(2)) * zr_se
    zu = atanh(r) + moe
    zl = atanh(r) - moe
    return tanh((zl, zu))

def rho_rxy_rxz(rxy, rxz, ryz):
    num = (ryz-1/2.*rxy*rxz)*(1-pow(rxy,2)-pow(rxz,2)-pow(ryz,2))+pow(ryz,3)
    den = (1 - pow(rxy,2)) * (1 - pow(rxz,2))
    return num/float(den)

def dependent_corr(xy, xz, yz, n, twotailed=True, conf_level=0.95, method='steiger'):
    """
    Calculates the statistic significance between two dependent correlation coefficients
    @param xy: correlation coefficient between x and y
    @param xz: correlation coefficient between x and z
    @param yz: correlation coefficient between y and z
    @param n: number of elements in x, y and z
    @param twotailed: whether to calculate a one or two tailed test, only works for 'steiger' method
    @param conf_level: confidence level, only works for 'zou' method
    @param method: defines the method uses, 'steiger' or 'zou'
    @return: t and p-val
    """
    if method == 'steiger':
        d = xy - xz
        determin = 1 - xy * xy - xz * xz - yz * yz + 2 * xy * xz * yz
        av = (xy + xz)/2
        cube = (1 - yz) * (1 - yz) * (1 - yz)

        t2 = d * np.sqrt((n - 1) * (1 + yz)/(((2 * (n - 1)/(n - 3)) * determin + av * av * cube)))
        p = 1 - t.cdf(abs(t2), n - 3)

        if twotailed:
            p *= 2

        return t2, p
    elif method == 'zou':
        L1 = rz_ci(xy, n, conf_level=conf_level)[0]
        U1 = rz_ci(xy, n, conf_level=conf_level)[1]
        L2 = rz_ci(xz, n, conf_level=conf_level)[0]
        U2 = rz_ci(xz, n, conf_level=conf_level)[1]
        rho_r12_r13 = rho_rxy_rxz(xy, xz, yz)
        lower = xy - xz - pow((pow((xy - L1), 2) + pow((U2 - xz), 2) - 2 * rho_r12_r13 * (xy - L1) * (U2 - xz)), 0.5)
        upper = xy - xz + pow((pow((U1 - xy), 2) + pow((xz - L2), 2) - 2 * rho_r12_r13 * (U1 - xy) * (xz - L2)), 0.5)
        return lower, upper
    else:
        raise Exception('Wrong method!')

In [3]:
NFA = 8
df_FA_behav = pd.read_csv(os.path.join(Dir_output,f'df_FA8_behav_merged.csv'))
df_loadings = pd.read_csv(os.path.join(Dir_output,f'FA_n{NFA}_loadings.csv'))
List_106vars = list(df_loadings['index'])
List_FAs = [f'FA{x}' for x in range(1,NFA+1)]

List_Y5 = [x for x in df_FA_behav.columns if ('_ACC_' in x) or ('_RT' in x) or x == 'Recog_Memory']
List_Y3 = [x for x in List_Y5 if 'SemanticD' not in x and 'Recog_Memory' not in x ]
List_Y4 = [x for x in List_Y5 if 'Recog_Memory' not in x]


dict_behav = {'RT_LD_V':['LexicalD_RT_V_ELP_z','LexicalD_RT_V_ECP_z','LexicalD_RT_V_BLP_z'],
                 'RT_LD_A':['LexicalD_RT_A_MALD_z','LexicalD_RT_A_AELP_z'],
                 'ACC_LD_V':['LexicalD_ACC_V_ELP','LexicalD_ACC_V_ECP','LexicalD_ACC_V_BLP'],
                 'ACC_LD_A':['LexicalD_ACC_A_MALD','LexicalD_ACC_A_AELP'],
                 'ACC_Naming':['Naming_ACC_ELP'],
                 'RT_Naming':['Naming_RT_ELP_z'],
                 'ACC_SD':['SemanticD_ACC_Calgary'],
                 'RT_SD':['SemanticD_RT_Calgary_z'],
                 'ACC_Recog':['Recog_Memory']
             }
dict_behav_combine = {'ACC(3tasks)':['ACC_LD_V','ACC_LD_A','ACC_Naming'],
                      'ACC(4tasks)':['ACC_LD_V','ACC_LD_A','ACC_Naming','ACC_SD'],
                      'ACC(5tasks)':['ACC_LD_V','ACC_LD_A','ACC_Naming','ACC_SD','ACC_Recog'],
                      'RT(3tasks)':['RT_LD_V','RT_LD_A','RT_Naming'],
                      'RT(4tasks)':['RT_LD_V','RT_LD_A','RT_Naming','RT_SD']}
# dict_behav_combine = {'ACC(3tasks)':['ACC_LD_V','ACC_LD_A','ACC_Naming'],
#                       'ACC(4tasks)':['ACC_LD_V','ACC_LD_A','ACC_Naming','ACC_SD'],
#                       'RT(3tasks)':['RT_LD_V','RT_LD_A','RT_Naming'],
#                       'RT(4tasks)':['RT_LD_V','RT_LD_A','RT_Naming','RT_SD']}


options = {'List_FAs':List_FAs,'List_Y4':List_Y4,'List_Y5':List_Y5,
          'dict_behav':dict_behav,'dict_behav_combine':dict_behav_combine,
          'df_FA_behav':df_FA_behav,
          'df_loadings':df_loadings
          }

In [4]:
def compare_r(options,bestN):
    List_Y = options['List_Y5']
    df_FA_behav = options['df_FA_behav']
    df_loadings = options['df_loadings']
    
    
    
    dfs_table = []
    for var in List_Y5:

        List_Y = [var]

        df_bestpred = df_FA_behav[List_106vars+List_FAs+List_Y].dropna()
        n = df_bestpred.shape[0]

        df_corr = df_bestpred.corr(method='spearman')

        df_table = []

        for iFA in range(1,9):
            List_var_tmp = list(df_loadings.loc[~df_loadings[f'Factor {iFA}'].isna(),'index'].values)
            df_r = df_corr.loc[List_var_tmp,List_Y].abs()
            df_rank = df_r.rank(ascending=False)

            best = df_rank.loc[df_rank[var]==bestN].index[0]
            best_r = df_r.loc[best,var]

            df_r = df_corr.loc[[f"FA{iFA}"]+List_var_tmp,List_Y].abs()
            df_rank = df_r.rank(ascending=False)

            FA_rank = df_rank.loc[f"FA{iFA}",var]
            FA_r = df_r.loc[f"FA{iFA}",var]
            df_table.append(pd.DataFrame({'variable':var,'best':best,'best r':best_r,'FA':f"FA{iFA}",'FA_rank':FA_rank,'FA_r':FA_r},index=[0]))

        df_table = pd.concat(df_table,ignore_index=True)

        for i, row in df_table.iterrows():

            x = row['variable']
            y = row['best']
            z = row['FA']
            if y!=z:

                corr_matrix = df_corr.loc[[x,y,z],[x,y,z]].abs()
                r_xy = corr_matrix.loc[x, y]  
                r_xz = corr_matrix.loc[x, z]  
                r_yz = corr_matrix.loc[y, z]  
        #             n = df_table.shape[0]  

                t2,p = dependent_corr(r_xy, r_xz, r_yz, n,twotailed=True,conf_level=0.95, method='steiger')
                df_table.loc[i,'t_steiger'] = t2
                df_table.loc[i,'p_steiger'] = p

        df_table['N'] = n
        dfs_table.append(df_table)
    dfs_table = pd.concat(dfs_table,ignore_index=True)
    dfs_table['p_steiger_FDR'] = stats.false_discovery_control(dfs_table['p_steiger'])
    return(dfs_table)


In [5]:
Nbest = 1

dfs_table = compare_r(options,Nbest)

# The factor scores outperform single variables
df = dfs_table.loc[dfs_table['t_steiger']<0].loc[dfs_table['p_steiger_FDR']<.05]
for i,row in df.iterrows():
    print(f"{row['FA']} is the {Nbest} best in predicting {row['variable']}")


# The single variables outperform factor scores 
dfs_table.loc[dfs_table['t_steiger']>0].loc[dfs_table['p_steiger_FDR']<.05]

FA7 is the 1 best in predicting LexicalD_RT_V_BLP_z
FA7 is the 1 best in predicting LexicalD_ACC_V_ELP
FA7 is the 1 best in predicting LexicalD_ACC_V_BLP


,variable,best,best r,FA,FA_rank,FA_r,t_steiger,p_steiger,N,p_steiger_FDR
1,LexicalD_RT_V_ELP_z,NLett,0.519464,FA2,6.0,0.435430,22.467356,0.000000e+00,11617,0.000000e+00
2,LexicalD_RT_V_ELP_z,Consistency_Token_FB_ON,0.040339,FA3,8.0,0.002943,6.354156,2.174145e-10,11617,5.671683e-10
3,LexicalD_RT_V_ELP_z,Consistency_Token_FF_R,0.197958,FA4,6.0,0.045782,16.582020,0.000000e+00,11617,0.000000e+00
5,LexicalD_RT_V_ELP_z,AoA_LWV,0.470621,FA6,3.0,0.236314,29.280698,0.000000e+00,11617,0.000000e+00
7,LexicalD_RT_V_ELP_z,Assoc_Freq_Token123,0.433700,FA8,5.0,0.286981,17.360112,0.000000e+00,11617,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...
111,Naming_ACC_ELP,Assoc_Freq_Type123,0.160876,FA8,5.0,0.104291,6.113763,1.004328e-09,11617,2.459579e-09
112,SemanticD_ACC_Calgary,Nsenses_WordNet,0.150814,FA1,15.0,0.028077,6.281706,3.956857e-10,2427,1.010261e-09
115,SemanticD_ACC_Calgary,Consistency_Type_FF_R,0.106698,FA4,7.0,0.052408,2.731856,6.343516e-03,2427,1.014963e-02
116,SemanticD_ACC_Calgary,TrigramF_Avg_C_Log,0.162491,FA5,6.0,0.064499,4.713177,2.576919e-06,2427,4.987585e-06


In [48]:
df = dfs_table.loc[~((dfs_table['t_steiger']>0) & (dfs_table['p_steiger_FDR']<.05))].copy()
df.loc[df['variable'].str.contains('_RT_'),'type'] = 'RT'
df.loc[~df['variable'].str.contains('_RT_'),'type'] = 'ACC'
df.loc[df['variable'].str.contains('LexicalD'),'task'] = 'LD'
df.loc[df['variable'].str.contains('SemanticD'),'task'] = 'SD'
df.loc[df['variable'].str.contains('Naming'),'task'] = 'Naming'
df.groupby(['FA','type','task']).count()

variable  best  best r  FA_rank  FA_r  t_steiger  p_steiger  \
FA  type task                                                                  
FA1 ACC  Naming         1     1       1        1     1          1          1   
    RT   LD             1     1       1        1     1          1          1   
FA2 ACC  LD             2     2       2        2     2          2          2   
         SD             1     1       1        1     1          1          1   
FA3 ACC  LD             2     2       2        2     2          2          2   
         SD             1     1       1        1     1          1          1   
    RT   LD             4     4       4        4     4          4          4   
         SD             1     1       1        1     1          1          1   
FA4 ACC  LD             2     2       2        2     2          2          2   
    RT   LD             2     2       2        2     2          2          2   
FA5 ACC  Naming         1     1       1        1     1          1          1   
    RT   LD             2     2       2        2     2          2          2   
         Naming         1     1       1        1     1          1          1   
         SD             1     1       1        1     1          1          1   
FA6 ACC  SD             1     1       1        1     1          1          1   
    RT   SD             1     1       1        1     1          1          1   
FA7 ACC  LD             5     5       5        5     5          5          5   
         Naming         1     1       1        1     1          1          1   
         SD             1     1       1        1     1          1          1   
    RT   LD             3     3       3        3     3          3          3   
         Naming         1     1       1        1     1          1          1   
FA8 ACC  LD             1     1       1        1     1          1          1   
    RT   LD             1     1       1        1     1          1          1   

                 N  p_steiger_FDR  
FA  type task                      
FA1 ACC  Naming  1              1  
    RT   LD      1              1  
FA2 ACC  LD      2              2  
         SD      1              1  
FA3 ACC  LD      2              2  
         SD      1              1  
    RT   LD      4              4  
         SD      1              1  
FA4 ACC  LD      2              2  
    RT   LD      2              2  
FA5 ACC  Naming  1              1  
    RT   LD      2              2  
         Naming  1              1  
         SD      1              1  
FA6 ACC  SD      1              1  
    RT   SD      1              1  
FA7 ACC  LD      5              5  
         Naming  1              1  
         SD      1              1  
    RT   LD      3              3  
         Naming  1              1  
FA8 ACC  LD      1              1  
    RT   LD      1              1

### The second best

In [9]:
Nbest = 2

dfs_table = compare_r(options,Nbest)

# The factor scores outperform single variables
df = dfs_table.loc[dfs_table['t_steiger']<0].loc[dfs_table['p_steiger_FDR']<.05]
for i,row in df.iterrows():
    print(f"{row['FA']} is the {Nbest} best in predicting {row['variable']}")


# The single variables outperform factor scores 
dfs_table.loc[dfs_table['t_steiger']>0].loc[dfs_table['p_steiger_FDR']<.05]

FA1 is the 2 best in predicting LexicalD_RT_V_ELP_z
FA7 is the 2 best in predicting LexicalD_RT_V_ELP_z
FA5 is the 2 best in predicting LexicalD_RT_V_ECP_z
FA7 is the 2 best in predicting LexicalD_RT_V_BLP_z
FA3 is the 2 best in predicting Recog_Memory
FA6 is the 2 best in predicting Recog_Memory
FA7 is the 2 best in predicting LexicalD_ACC_V_ELP
FA7 is the 2 best in predicting LexicalD_ACC_V_BLP
FA2 is the 2 best in predicting LexicalD_ACC_A_AELP
FA1 is the 2 best in predicting Naming_ACC_ELP
FA5 is the 2 best in predicting Naming_ACC_ELP


,variable,best,best r,FA,FA_rank,FA_r,t_steiger,p_steiger,N,p_steiger_FDR
1,LexicalD_RT_V_ELP_z,NPhon,0.481339,FA2,6.0,0.435430,12.597197,0.000000e+00,11617,0.000000e+00
2,LexicalD_RT_V_ELP_z,Consistency_Type_FB_ON,0.037025,FA3,8.0,0.002943,5.819115,6.072278e-09,11617,1.777252e-08
3,LexicalD_RT_V_ELP_z,Consistency_Type_FF_R,0.193558,FA4,6.0,0.045782,16.090698,0.000000e+00,11617,0.000000e+00
5,LexicalD_RT_V_ELP_z,AoA_Kuper,0.459814,FA6,3.0,0.236314,27.444722,0.000000e+00,11617,0.000000e+00
7,LexicalD_RT_V_ELP_z,Assoc_Freq_Type123,0.429567,FA8,5.0,0.286981,16.807937,0.000000e+00,11617,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...
109,Naming_ACC_ELP,AoA_Kuper,0.208888,FA6,4.0,0.103916,11.762263,0.000000e+00,11617,0.000000e+00
111,Naming_ACC_ELP,Assoc_Freq_Token123,0.160738,FA8,5.0,0.104291,6.110511,1.024941e-09,11617,3.153666e-09
112,SemanticD_ACC_Calgary,AoA_Kuper,0.112081,FA1,15.0,0.028077,4.464284,8.401478e-06,2427,1.976818e-05
115,SemanticD_ACC_Calgary,Consistency_Token_FF_R,0.104060,FA4,7.0,0.052408,2.599825,9.383751e-03,2427,1.521689e-02


In [10]:
Nbest = 3

dfs_table = compare_r(options,Nbest)

# The factor scores outperform single variables
df = dfs_table.loc[dfs_table['t_steiger']<0].loc[dfs_table['p_steiger_FDR']<.05]
for i,row in df.iterrows():
    print(f"{row['FA']} is the {Nbest} best in predicting {row['variable']}")


# The single variables outperform factor scores 
dfs_table.loc[dfs_table['t_steiger']>0].loc[dfs_table['p_steiger_FDR']<.05]

FA1 is the 3 best in predicting LexicalD_RT_V_ELP_z
FA6 is the 3 best in predicting LexicalD_RT_V_ELP_z
FA7 is the 3 best in predicting LexicalD_RT_V_ELP_z
FA3 is the 3 best in predicting LexicalD_RT_V_ECP_z
FA5 is the 3 best in predicting LexicalD_RT_V_ECP_z
FA1 is the 3 best in predicting LexicalD_RT_V_BLP_z
FA3 is the 3 best in predicting LexicalD_RT_V_BLP_z
FA7 is the 3 best in predicting LexicalD_RT_V_BLP_z
FA3 is the 3 best in predicting LexicalD_RT_A_AELP_z
FA5 is the 3 best in predicting Naming_RT_ELP_z
FA6 is the 3 best in predicting Naming_RT_ELP_z
FA3 is the 3 best in predicting SemanticD_RT_Calgary_z
FA6 is the 3 best in predicting SemanticD_RT_Calgary_z
FA3 is the 3 best in predicting Recog_Memory
FA6 is the 3 best in predicting Recog_Memory
FA1 is the 3 best in predicting LexicalD_ACC_V_ELP
FA7 is the 3 best in predicting LexicalD_ACC_V_ELP
FA1 is the 3 best in predicting LexicalD_ACC_V_ECP
FA2 is the 3 best in predicting LexicalD_ACC_V_ECP
FA1 is the 3 best in predicting

,variable,best,best r,FA,FA_rank,FA_r,t_steiger,p_steiger,N,p_steiger_FDR
1,LexicalD_RT_V_ELP_z,OLD20,0.469997,FA2,6.0,0.435430,11.003079,0.000000e+00,11617,0.000000e+00
2,LexicalD_RT_V_ELP_z,Consistency_Type_FB,0.029239,FA3,8.0,0.002943,10.546635,0.000000e+00,11617,0.000000e+00
3,LexicalD_RT_V_ELP_z,Consistency_Token_FF,0.113207,FA4,6.0,0.045782,26.013609,0.000000e+00,11617,0.000000e+00
7,LexicalD_RT_V_ELP_z,Assoc_Freq_Token,0.392000,FA8,5.0,0.286981,12.800105,0.000000e+00,11617,0.000000e+00
9,LexicalD_RT_V_ECP_z,NPhon,0.363979,FA2,11.0,0.200888,31.713623,0.000000e+00,7449,0.000000e+00
11,LexicalD_RT_V_ECP_z,Consistency_Token_FF_N,0.035225,FA4,7.0,0.011257,3.643305,2.709908e-04,7449,5.511677e-04
13,LexicalD_RT_V_ECP_z,Mink_Perceptual_Lanc,0.275475,FA6,4.0,0.212056,6.779423,1.298828e-11,7449,7.084515e-11
15,LexicalD_RT_V_ECP_z,Assoc_Freq_Type,0.531204,FA8,5.0,0.427146,12.638327,0.000000e+00,7449,0.000000e+00
17,LexicalD_RT_V_BLP_z,NPhon,0.265277,FA2,12.0,0.091665,26.728923,0.000000e+00,9608,0.000000e+00
19,LexicalD_RT_V_BLP_z,Consistency_Type_FF,0.032713,FA4,5.0,0.025148,2.914796,3.567430e-03,9608,6.204226e-03
